In [1]:
import sys

import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__)

3.14.0 2.4.4 3.0.2 1.9.1


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 3101

data = pd.read_csv("data/heart_disease.csv")
feature_names = [c for c in data.columns if c != "disease"]
X, y = data[feature_names].to_numpy(), data["disease"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)
print(data.shape, round(y.mean(), 3), X_train.shape, X_test.shape)

(297, 14) 0.461 (207, 13) (90, 13)


Part 1: Entropy and Information Gain by Hand1. 
Parent Entropy
Total patients: 
12Class distribution: 
6 patients with disease = 1, 6 patients with disease = 0.
Proportions: $p_1 = \frac{6}{12} = 0.5$, $p_0 = \frac{6}{12} = 0.5$.
Formula & Calculation:$$H(\text{parent}) = - \sum p_c \log_2 p_c = - \left( 0.5 \log_2 0.5 + 0.5 \log_2 0.5 \right) = -(-0.5 - 0.5) = 1.0 \text{ bit}$$

2. Root
To find the information gain for each feature, we split the 12 patients by whether the feature equals 1 or 0, compute the entropy of each child node, and calculate the size-weighted average entropy.
Feature 1: age_55_plusage_55_plus = 1 (7 patients: p1, p2, p3, p4, p5, p7, p6? Let's check: p1-p5 are 1, p7 is 1 $\rightarrow$ 6 patients with age=1? Wait:p1: 1, p2: 1, p3: 1, p4: 1, p5: 1, p6: 0, p7: 1, p8: 0, p9: 0, p10: 0, p11: 0, p12: 0.Count age_55_plus = 1: p1, p2, p3, p4, p5, p7 $\rightarrow$ 6 patients.Out of these 6: p1, p2, p3, p4, p5 have disease=1; p7 has disease=0. So left child: 5 ones, 1 zero. Entropy $H(\text{left}) = -(\frac{5}{6}\log_2\frac{5}{6} + \frac{1}{6}\log_2\frac{1}{6}) \approx 0.650$.age_55_plus = 0 (6 patients: p6, p8, p9, p10, p11, p12): p6 has disease=1; p8, p9, p10, p11, p12 have disease=0. So right child: 1 one, 5 zeros. Entropy $H(\text{right}) \approx 0.650$.Weighted entropy: $\frac{6}{12}(0.650) + \frac{6}{12}(0.650) = 0.650$.Information Gain: $1.0 - 0.650 = 0.350$ bits
Feature 2: exercise_anginaexercise_angina = 1 (4 patients: p1, p2, p3, p4): all 4 have disease=1 ($H = 0$).exercise_angina = 0 (8 patients: p5–p12): 2 have disease=1 (p5, p6), 6 have disease=0.Right child proportions: 2 ones, 6 zeros ($\frac{2}{8}=0.25$, $\frac{6}{8}=0.75$).$H(\text{right}) = -(0.25\log_2 0.25 + 0.75\log_2 0.75) \approx 0.811$.Weighted entropy: $\frac{4}{12}(0) + \frac{8}{12}(0.811) \approx 0.541$.Information Gain: $1.0 - 0.541 = 0.459$ bits.Feature 3: malemale = 1 (6 patients: p1, p2, p3, p7, p8, p9): 3 have disease=1 (p1, p2, p3), 3 have disease=0 (p7, p8, p9). $H = 1.0$.male = 0 (6 patients: p4, p5, p6, p10, p11, p12): 3 have disease=1 (p4, p5, p6), 3 have disease=0. $H = 1.0$.Information Gain: $1.0 - 1.0 = 0.0$ bits.Feature 4: high_bphigh_bp = 1 (6 patients: p1, p2, p4, p5, p10, p11): 4 have disease=1 (p1, p2, p4, p5), 2 have disease=0 (p10, p11).$H = -(\frac{4}{6}\log_2\frac{4}{6} + \frac{2}{6}\log_2\frac{6}{6}) \approx 0.918$.high_bp = 0 (6 patients: p3, p6, p7, p8, p9, p12): 2 have disease=1 (p3, p6), 4 have disease=0.$H \approx 0.918$.Information Gain: $1.0 - 0.918 = 0.082$ bits.Root Choice: exercise_angina has the highest information gain (0.459 bits), so the tree chooses it at the root.